In [ ]:
%load_ext autoreload
%autoreload 2

# Nash-DQN and SRE-DQN Training

Train the Nash and SRE models and save checkpoints into timestamped `pt_files/` folders.

In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import torch

from NashRL import run_Nash_Agent, run_training_loop
from NashAgent_lib import NashNN
from sre_agent import SreNN
from experiment_config import seed_everything

np.set_printoptions(precision=4)

print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Shared Market Environment

Identical parameters to `NashDQN-Training.ipynb`. Both agents are trained on this same environment.

In [ ]:
from experiment_config import (
    NUM_PLAYERS as num_players, sim_dict, norm_mean, norm_std, T, make_sim_obj,
)

sim_obj = make_sim_obj()

print('Agents:', num_players, '| T=', sim_dict['T'].item(), '| dt=', sim_dict['dt'].item())


## Training Configuration


In [ ]:
from experiment_config import (
    seed_everything, eps_slug,
    BASE_SEED, MAX_STEPS,
    SRE_EPS_LIST, SRE_EPS_REG, SRE_DELTA_MIN, SRE_GAMMA,
    NET_KWARGS,
)

# Training-specific config
NASH_SEED = BASE_SEED
SRE_TRAIN_SEED = BASE_SEED

NUM_SIM    = 20000
RV_MIN     = 0.5
RV_MAX     = 2.5
EARLY_STOP = True
EARLY_LIM  = 2000

PRIMARY_SRE_EPS = 0.5
SRE_EPS_DECAY   = None

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M')

NASH_MODEL_DIR = os.path.join('pt_files', f'nash_{RUN_TAG}')
SRE_MODEL_DIRS = {
    eps: os.path.join('pt_files', f'sre_eps_{eps_slug(eps)}_{RUN_TAG}')
    for eps in SRE_EPS_LIST
}

seed_everything(BASE_SEED)

print('Nash model dir:', NASH_MODEL_DIR)
print('Seeds | base:', BASE_SEED, '| nash:', NASH_SEED, '| sre train:', SRE_TRAIN_SEED)
print('NUM_SIM:', NUM_SIM, '| MAX_STEPS:', MAX_STEPS)
print('SRE eps sweep:', SRE_EPS_LIST, '| decay horizon:', SRE_EPS_DECAY)


## Train Nash-DQN

In [ ]:

os.makedirs(NASH_MODEL_DIR, exist_ok=True)

seed_everything(NASH_SEED)
nash_agent = NashNN(**NET_KWARGS)

start = time.time()
nash_agent, nash_loss = run_Nash_Agent(
    sim_obj, sim_dict, MAX_STEPS,
    nash_agent=nash_agent,
    num_sim=NUM_SIM,
    norm_mean=norm_mean,
    norm_std=norm_std,
    rv_min=RV_MIN,
    rv_max=RV_MAX,
    early_stop=EARLY_STOP,
    early_lim=EARLY_LIM,
    path=os.path.join(NASH_MODEL_DIR, ''),
    AN_file_name=os.path.join(NASH_MODEL_DIR, 'Action_Net'),
    VN_file_name=os.path.join(NASH_MODEL_DIR, 'Value_Net'),
)
nash_train_time = time.time() - start
print(f'Nash-DQN training time: {nash_train_time:.1f}s')


## Train SRE-DQN

In [ ]:

def make_sre_action(agent, eps_b, noise_std):
    def fn(cur_s, cur_ivt):
        mu_sr = agent.compute_sre_action(cur_s, cur_ivt, eps_b)
        return mu_sr + torch.randn_like(mu_sr) * noise_std
    return fn


def make_eps_schedule(eps_0):
    def eps_schedule(k, num_sim):
        if SRE_EPS_DECAY is None:
            return eps_0
        return eps_0 * max(0.0, 1.0 - k / max(SRE_EPS_DECAY, 1))
    return eps_schedule


sre_agents = {}
sre_losses = {}
sre_train_times = {}

for eps in SRE_EPS_LIST:
    eps_dir = SRE_MODEL_DIRS[eps]
    os.makedirs(eps_dir, exist_ok=True)

    seed_everything(SRE_TRAIN_SEED)
    sre_agent = SreNN(
        **NET_KWARGS,
        eps_reg=SRE_EPS_REG,
        delta_min=SRE_DELTA_MIN,
        gamma=SRE_GAMMA,
    )

    start = time.time()
    trained_agent, train_loss = run_training_loop(
        sim_obj=sim_obj,
        sim_dict=sim_dict,
        max_steps=MAX_STEPS,
        agent=sre_agent,
        make_action_fn=make_sre_action,
        eps_schedule_fn=make_eps_schedule(eps),
        num_sim=NUM_SIM,
        norm_mean=norm_mean,
        norm_std=norm_std,
        rv_min=RV_MIN,
        rv_max=RV_MAX,
        early_stop=EARLY_STOP,
        early_lim=EARLY_LIM,
        path=os.path.join(eps_dir, ''),
        AN_file_name=os.path.join(eps_dir, f'SRE_Action_Net_eps_{eps_slug(eps)}'),
        VN_file_name=os.path.join(eps_dir, f'SRE_Value_Net_eps_{eps_slug(eps)}'),
        checkpoint_metadata={
            'trainer': 'locally_linear_quadratic_sre',
            'eps_0': float(eps),
            'eps_decay_horizon': None if SRE_EPS_DECAY is None else int(SRE_EPS_DECAY),
            'eps_reg': float(SRE_EPS_REG),
            'delta_min': float(SRE_DELTA_MIN),
            'gamma': float(SRE_GAMMA),
        },
        desc=f'SRE-DQN eps={eps:g}',
    )
    sre_agents[eps] = trained_agent
    sre_losses[eps] = train_loss
    sre_train_times[eps] = time.time() - start
    print(f'SRE-DQN training time (eps={eps:g}): {sre_train_times[eps]:.1f}s')
